## 读者模式：CONSTRUCTED 月度因子归因

本 Notebook 的默认运行是构造数据方法演示。所有新计算值均为 CONSTRUCTED，不是市场观测，也不是对历史研究的重新验证。French 原始、解压和处理数据未随包提供；French 执行分支跳过。A10-A 仅有限范围接受；A10-A.1 协议已技术接受，但仍待权限与执行授权。G01 权利、主体和用途未知；G05 未授权 A10-B。

下文保留课程方法说明；当前运行内容以读者模式代码和标签为准。不要重训 Project 6，也不要重建 Chapter 39 的历史行。

# Project 2：CAPM 与 Fama–French 因子回归

本 Notebook 默认对一个构造资产估计 CAPM、FF3、FF5 三个模型。输入为种子 102 生成的 120 个月因子序列，不是 NoDur、HiTec、Enrgy 等行业市场数据。该演示用于学习同月超额收益、条件 Beta、模型截距、HAC 推断和残差。

三种规格使用同一组构造因子列，因此本次构造模型严格嵌套。模型结果是方法演示，不是因果效应、未来收益或可交易优势。历史报告另行保留 2010–2019 French 快照上的三个行业 × 三种模型共九组回归；那是不同数据和不同证据身份，不是本 Notebook 的新输出。

本项目使用资本资产定价模型（Capital Asset Pricing Model, CAPM）、Fama–French 三因子模型（Fama–French Three-Factor Model, FF3）与五因子模型（Fama–French Five-Factor Model, FF5）。普通最小二乘法（Ordinary Least Squares, OLS）估计系数；异方差与自相关一致协方差估计（Heteroskedasticity and Autocorrelation Consistent Covariance Estimation, HAC）用于推断；置信区间（Confidence Interval, CI）描述逐项不确定性。原理与手算见第 8、15 章。

另见[项目说明](../../../../../projects/project02/README.md)、[读者指南](../../../../../projects/project02/READER_GUIDE.md)、[历史研究报告](../../../../../projects/project02/research_report/report.md)、[练习](../../../../../projects/project02/exercises.md)、[答案](../../../../../projects/project02/solutions.md)和[数据字典](../../../../../projects/project02/data_dictionary.md)。

## 构造样本、方程和时间

月度索引从 2010-01 起，共 120 个月。随机种子为 102；五列因子由 N(0, 0.03²) 伪随机抽样产生，RF 固定为 0.001。单一构造资产总收益为：

\[
R_t=RF_t+0.002+1.1\,f_{\mathrm{MKT},t}+0.4\,f_{\mathrm{SMB},t}+\epsilon_t,\qquad
\epsilon_t\sim N(0,0.015^2).
\]

目标超额收益为构造资产总收益减同月 RF 一次：\(y_t=asset_t-RF_t\)。它们都是小数月收益，不是 French 发布观测。

- CAPM 使用 Mkt-RF；FF3 使用 Mkt-RF、SMB、HML；FF5 使用 Mkt-RF、SMB、HML、RMW、CMA。三者从同一构造因子矩阵取列，因此本次模型严格嵌套。
- Mkt-RF 等名称在这里是构造列名，不表示使用了官方 French 因子。RF 只从资产总收益中扣一次。
- 所有三个主模型使用 HAC 6。仅 FF3 另按 HAC 3、6、12 计算推断；更换带宽会改变标准误、区间和 p 值，不改变 OLS 点估计。
- 代码确认每个模型有 120 个观测，并检查超额收益 = 拟合值 + 残差。Notebook 不计算或断言样本方差恒等式。
- 这些月份只是构造数组的索引；它们不是历史公告、信号或成交时间。Notebook 没有时间顺序预测或交易测试。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project02_factor_regression_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from quant_course.factor_models import fit_monthly_factor_model
print("CONSTRUCTED factor attribution; French snapshot/alignment verification SKIPPED: rights unknown")


In [ ]:
rng = np.random.default_rng(102)
idx = pd.period_range("2010-01", periods=120, freq="M", name="month")
factors = pd.DataFrame(rng.normal(0, 0.03, (120,5)), index=idx, columns=["Mkt-RF","SMB","HML","RMW","CMA"])
factors["RF"] = 0.001
asset = factors["RF"] + 0.002 + 1.1*factors["Mkt-RF"] + 0.4*factors["SMB"] + rng.normal(0,0.015,120)
models = {"CAPM": ("Mkt-RF",), "FF3": ("Mkt-RF","SMB","HML"), "FF5": ("Mkt-RF","SMB","HML","RMW","CMA")}
fits = {m:fit_monthly_factor_model(asset,factors,cols,hac_maxlags=6) for m,cols in models.items()}
assert all(f.nobs == 120 for f in fits.values())


## 三个构造模型的结果

本单元显示三个模型的 OLS + 月度 HAC(6) 系数表。每表只有本模型估计的参数行：CAPM 为 const、Mkt-RF；FF3 为 const、Mkt-RF、SMB、HML；FF5 为 const、Mkt-RF、SMB、HML、RMW、CMA。这里不是历史九组行业回归；没有跨资产 Holm 校正，不能将任何值解释为历史 Alpha。

表中 estimate 是系数估计；standard_error_hac、t_hac 和 p_hac 分别为 HAC 标准误、t 统计量和 p 值；ci_low_hac、ci_high_hac 为 95% 区间端点。const 是模型截距，因子行是条件载荷。遗漏因子不产生行，也不表示估计值为零。

每个模型的 CSV 以 index=False 写出，所以参数名索引未保存在文件中。请用文件名和上面的固定行序恢复每行含义；六个 CSV 列仍按 estimate、standard_error_hac、t_hac、p_hac、ci_low_hac、ci_high_hac 排列。表中构造结果不得替代历史报告中的结果。

In [ ]:
for model, fit in fits.items():
    print(model, "CONSTRUCTED OLS + monthly HAC(6)")
    display(fit.coefficient_table)
    fit.coefficient_table.to_csv(RUN_DIR / (model + "_coefficients_CONSTRUCTED.csv"), index=False)


## HAC 带宽与样本方差

更改 HAC 带宽会改变标准误、区间和 p 值，不会改变 OLS 点估计。对同一 120 个月、ddof=1 的 OLS 投影，样本方差恒等式为：\(\operatorname{Var}(y)=\operatorname{Var}(\hat y)+\operatorname{Var}(e)\)。这是理论上的样本内方差会计关系；本 Notebook 没有计算或断言该分解。

实际代码断言每个拟合有 120 个观测，并用 assert_allclose 检查超额收益 = fitted_excess + residuals。重构等式不等于方差分解验证。即使成立，这一 OLS 样本内关系也不表示能够预测未来风险，或把相关因子拆成互相独立的风险份额。

In [ ]:
for lag in (3, 6, 12):
    f = fit_monthly_factor_model(asset, factors, models["FF3"], hac_maxlags=lag)
    print("HAC bandwidth", lag, "months; R2", f.r_squared)
    display(f.coefficient_table)
fit = fits["FF3"]
np.testing.assert_allclose(asset - factors["RF"], fit.fitted_excess + fit.residuals)


## 图形阅读

本次默认 Notebook 只生成下方这一幅图，不生成历史报告中的五幅研究图。

- 构造 FF3 残差图：横轴标签为 Month，但代码仅向plot传入residuals.to_numpy()，未传日期横轴，Matplotlib遂使用 0–119 的观测序号，不是日历月份。纵轴标签为 Decimal excess return；实际绘图值是 FF3 超额收益残差，单位为小数月收益。它不是资产总收益，也不是未拟合的超额收益。
- 五幅历史图——Alpha、因子载荷、方差分解、历史残差、时间顺序归因误差——及各轴解释见[历史研究报告](../../../../../projects/project02/research_report/report.md)。它们不是本次构造 Notebook 的输出。

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(fits["FF3"].residuals.to_numpy())
ax.set(title="CONSTRUCTED FF3 residuals", xlabel="Month", ylabel="Decimal excess return")
fig.tight_layout(); fig.savefig(RUN_DIR / "project02_residuals_CONSTRUCTED.png", dpi=130)
plt.show()


## 来源与解释边界

以下官方页面用于历史数据定义和研究背景：[Kenneth R. French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)、[10 Industry Portfolios](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_10_ind_port.html)、[FF3 因子说明](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_factors.html)、[FF5 因子说明](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html)。原始研究背景见 [Fama and French (1993)](https://www.sciencedirect.com/science/article/pii/0304405X93900235) 与 [Fama and French (2015)](https://www.sciencedirect.com/science/article/pii/S0304405X14002323)。历史来源下载时间、CRSP 版本、路径与 SHA-256 记录在 projects/project02/data_manifest.json；它们不表示市场数据包含在本包中，也不构成使用授权。

历史官方 FF3 与 FF5 的 SMB 构造不同；本地共同 Mkt-RF、SMB 序列也有差异，故原历史回归不是严格嵌套比较。SMB 构造差异有官方说明，月度底层组合差异未逐项复算；Mkt-RF 文件数值差异原因未查明。当前构造 Notebook 把同一组生成因子列用于三个规格，所以其嵌套关系不同于官方历史文件。

本 Notebook 的 HAC 带宽只是推断协方差选择，不能弥补模型遗漏或证明市场因果效应。没有交易账本，成本、融资和借券不适用；不是零成本分析。French 原始、解压和处理数据未随包提供。A10-A 设计仅有限范围接受；A10-A.1 执行协议技术接受但待权限与执行授权；G01 权利、主体、用途未知，G05 未授权 A10-B。公开页面可访问不代表获准执行或再分发。